In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
from pyspark.sql.functions import col, count, lpad, row_number
from pyspark.sql.window import Window

In [ ]:
dbutils.widgets.dropdown("target", "all", [
    "all", "dim_fecha", "dim_estacion_aire", "dim_punto_trafico",
    "dim_distrito", "dim_magnitud",
])
dbutils.widgets.text("year_start", "2025")
dbutils.widgets.text("year_end", "2025")

TARGET = dbutils.widgets.get("target")
YEAR_START = int(dbutils.widgets.get("year_start"))
YEAR_END = int(dbutils.widgets.get("year_end"))
NOTEBOOK = "gold/dimensions"

errors = []
built = []

In [ ]:
# Calendar for the loaded range
if TARGET in ("dim_fecha", "all"):
    try:
        holidays_sql = ", ".join(f"'{d}'" for d in HOLIDAYS_ES_MMDD)
        dim_fecha = spark.sql(f"""
            SELECT fecha,
                   year(fecha) AS ano,
                   month(fecha) AS mes,
                   day(fecha) AS dia,
                   dayofweek(fecha) AS dia_semana,
                   date_format(fecha, 'MMM') AS nombre_mes,
                   dayofweek(fecha) IN (1, 7) AS es_finde,
                   date_format(fecha, 'MM-dd') IN ({holidays_sql}) AS es_festivo
            FROM (
                SELECT explode(sequence(
                    to_date('{YEAR_START}-01-01'),
                    to_date('{YEAR_END}-12-31'),
                    interval 1 day
                )) AS fecha
            )
        """)

        if not write_gold(dim_fecha, "dim_fecha"):
            raise Exception("write_gold returned False")
        built.append("dim_fecha")
        print(f"dim_fecha: {dim_fecha.count()} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail dim_fecha: {type(e).__name__}: {e}")

In [ ]:
# Promote station dim from silver
if TARGET in ("dim_estacion_aire", "all"):
    try:
        dim_estacion = (
            spark.table(f"{SILVER_TABLE}.estaciones_aire")
            .withColumn("codigo_corto", col("codigo_corto").cast("int"))
            .withColumn("cod_dis", lpad(col("cod_dis").cast("string"), 2, "0"))
        )

        if not write_gold(dim_estacion, "dim_estacion_aire"):
            raise Exception("write_gold returned False")
        built.append("dim_estacion_aire")
        print(f"dim_estacion_aire: {dim_estacion.count()} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail dim_estacion_aire: {type(e).__name__}: {e}")

In [ ]:
# Dedup on latest snapshot month
if TARGET in ("dim_punto_trafico", "all"):
    try:
        latest = Window.partitionBy("id").orderBy(col("partition_key").desc())
        dim_punto = (
            spark.table(f"{BRONZE_TABLE}.trafico_puntos_medida")
            .withColumn("_rn", row_number().over(latest))
            .filter(col("_rn") == 1)
            .drop("_rn")
            .withColumn("id", col("id").cast("int"))
            .withColumn("distrito", lpad(col("distrito").cast("string"), 2, "0"))
        )

        if not write_gold(dim_punto, "dim_punto_trafico"):
            raise Exception("write_gold returned False")
        built.append("dim_punto_trafico")
        print(f"dim_punto_trafico: {dim_punto.count()} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail dim_punto_trafico: {type(e).__name__}: {e}")

In [ ]:
# District coverage and centroids
if TARGET in ("dim_distrito", "all"):
    try:
        n_estaciones = (
            spark.table(f"{GOLD_TABLE}.dim_estacion_aire")
            .filter(col("cod_dis").isNotNull())
            .groupBy("cod_dis")
            .agg(count("*").alias("n_estaciones_aire"))
        )
        n_puntos = (
            spark.table(f"{GOLD_TABLE}.dim_punto_trafico")
            .filter(col("distrito").isNotNull())
            .groupBy(col("distrito").alias("cod_dis"))
            .agg(count("*").alias("n_puntos_trafico"))
        )

        dim_distrito = (
            spark.table(f"{BRONZE_TABLE}.distritos")
            .withColumn("cod_dis", lpad(col("cod_dis").cast("string"), 2, "0"))
            .join(n_estaciones, "cod_dis", "left")
            .join(n_puntos, "cod_dis", "left")
            .fillna({"n_estaciones_aire": 0, "n_puntos_trafico": 0})
            .withColumn("cobertura_aire", col("n_estaciones_aire") > 0)
            .selectExpr(
                "*",
                "ST_X(ST_Centroid(ST_GeomFromWKT(geometry))) AS centroid_lon",
                "ST_Y(ST_Centroid(ST_GeomFromWKT(geometry))) AS centroid_lat",
            )
        )

        if not write_gold(dim_distrito, "dim_distrito"):
            raise Exception("write_gold returned False")
        built.append("dim_distrito")
        print(f"dim_distrito: {dim_distrito.count()} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail dim_distrito: {type(e).__name__}: {e}")

In [ ]:
# Static gas catalogue
if TARGET in ("dim_magnitud", "all"):
    try:
        dim_magnitud = spark.createDataFrame(
            [(int(c), l) for c, l in MAGNITUD_LABELS.items()],
            "codigo INT, magnitud STRING",
        )

        if not write_gold(dim_magnitud, "dim_magnitud"):
            raise Exception("write_gold returned False")
        built.append("dim_magnitud")
        print(f"dim_magnitud: {dim_magnitud.count()} rows")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail dim_magnitud: {type(e).__name__}: {e}")

In [ ]:
print(f"dimensions (target={TARGET}) built={built} failed={len(errors)}")
log_errors(errors)